# 4.8 Lab - Adding ML to Engineering Workflows with DABs
The data science team has a script that trains a revenue model (`src/ml/train_revenue_model.py`). Make it part of the deployed workflow: train **only after** the engineering task has rebuilt `daily_summary`, in its own **serverless environment** with the ML libraries (`scikit-learn`, `mlflow`).

Starter: **`04-automated-deployment-dabs/labs/4.8_ml_workflow/`**. Complete `resources/ml_job.yml` (TODO 1 and 2 are in the file). Solution: `solutions/4.8_ml_workflow/`.

| Concept | Where |
|---|---|
| Task dependency (`depends_on`) | the ML task waits for `build_daily_summary` |
| Job **environments** (`environments` + `environment_key`) | one environment per set of libraries: the wheel for engineering, `scikit-learn` + `mlflow` for ML |
| Python script task (`spark_python_task`) | on serverless, `environment_key` works for script, wheel and dbt tasks. A **notebook** task can't take one; it installs libraries with `%pip` instead |
| Model registry in Unity Catalog | the script registers `<catalog>.<schema>.revenue_model` |

![orders_ml_job DAG: build_daily_summary wheel task then train_revenue_model script task, each with its own environment](../images/4.8-ml-job-dag.svg)

**How to read it:** this is the job your finished `resources/ml_job.yml` describes. TODO 1 is the blue part (two tasks and the `depends_on`
arrow), TODO 2 the dashed environments linked by `environment_key`. The checks below test exactly these links; after a
workspace run the model appears in Unity Catalog with a new version.

In [ ]:
from adelab import bundles
from adelab.spark import PROJECT_ROOT

LAB = PROJECT_ROOT / "04-automated-deployment-dabs" / "labs" / "4.8_ml_workflow"
status = bundles.workspace_status()
print((LAB / "resources" / "ml_job.yml").read_text())
print((LAB / "src" / "ml" / "train_revenue_model.py").read_text())

## CHECK (offline)

In [ ]:
assert not bundles.validate_schema(LAB)
cfg = bundles.preview(LAB, "dev", show=False)
job = cfg["resources"]["jobs"].get("orders_ml_job")
assert job, "TODO 1: job orders_ml_job"
tasks = {t["task_key"]: t for t in job["tasks"]}
assert {"build_daily_summary", "train_revenue_model"} <= set(tasks), f"TODO 1: tasks found: {sorted(tasks)}"
train = tasks["train_revenue_model"]
assert {"task_key": "build_daily_summary"} in train.get("depends_on", []), "TODO 1: train must depend on build_daily_summary"
assert "notebook_task" not in train, "TODO 1: use a spark_python_task (notebook tasks can't use environment_key)"
script = train.get("spark_python_task", {})
assert script.get("python_file", "").endswith("src/ml/train_revenue_model.py"), f"TODO 1: python_file is {script.get('python_file')}"
assert script.get("parameters") == ["--catalog", "workspace", "--schema", "orders_cicd_dev"],     f"TODO 1: parameters resolve to {script.get('parameters')}"
envs = {e["environment_key"]: e["spec"].get("dependencies", []) for e in job.get("environments", [])}
assert tasks["build_daily_summary"].get("environment_key") == "default" and "../dist/*.whl" in envs.get("default", []),     "TODO 2: environment 'default' with the wheel"
assert train.get("environment_key") == "ml" and {"scikit-learn", "mlflow"} <= set(envs.get("ml", [])),     "TODO 2: environment 'ml' with scikit-learn and mlflow, used by train_revenue_model"
print("All offline checks passed")

## Deploy and run (workspace)

In [ ]:
if bundles.require_workspace(status):
    bundles.cli("bundle", "deploy", cwd=LAB)
    bundles.cli("bundle", "run", "orders_ml_job", cwd=LAB)

Open *Catalog → workspace → orders_cicd_dev → Models*: `revenue_model` is registered with a new version per run. Clean up:

In [ ]:
if bundles.require_workspace(status):
    bundles.cli("bundle", "destroy", "--auto-approve", cwd=LAB)